<a href="https://colab.research.google.com/github/kuds/reinforce-tactics/blob/main/notebooks/ppo_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# Force headless pygame BEFORE anything imports pygame (directly or
# transitively). SDL picks its video driver at import time, so
# setting this later is a no-op for an already-loaded SDL backend.
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")

# Reinforce Tactics — PPO Training

This notebook trains a **MaskablePPO** agent against the **`RandomBot`**
opponent on the 6×6 beginner map and records reference metrics at five
training checkpoints:

| Checkpoint | Timesteps |
|------------|-----------|
| 1 | 10,000 |
| 2 | 50,000 |
| 3 | 200,000 |
| 4 | 1,000,000 |
| 5 | 2,000,000 |

We use the random opponent as the **starting curriculum**: it is weak enough
that the agent can experience early wins (and the +win terminal reward)
within only a few thousand steps. Once the agent reliably beats `RandomBot`,
graduate to the harder `SimpleBot` opponent by setting `OPPONENT = "bot"`
in the config cell.

At each checkpoint the agent is evaluated over **50 episodes** and we record:
- **Win rate** (% of games won against the configured opponent)
- **Average episode reward**
- **Average episode length** (steps)

The goal is to provide a **reference curve** so that users can run the same
notebook and compare their results to known-good training runs.

**Runtime:** GPU recommended (L4 or better). CPU is possible but slower.

---

### Why MaskablePPO?

The game has a `MultiDiscrete` action space where many action combinations
are invalid at any given time (e.g. you can't attack a tile with no enemy).
**Action masking** prevents the agent from sampling these invalid actions,
which typically yields 2–3× faster convergence compared to plain PPO.

## 1. Setup

In [ ]:
# Install dependencies
!pip install -q gymnasium stable-baselines3 sb3-contrib tensorboard pandas numpy torch matplotlib opencv-python-headless

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Clone repo and install as a package
import os
import sys
from pathlib import Path

REPO_DIR = Path("reinforce-tactics")
if REPO_DIR.exists():
    os.chdir(REPO_DIR)
elif Path("notebooks").exists():
    # Already inside the repo
    os.chdir("..")
else:
    print("Cloning repository...")
    !git clone https://github.com/kuds/reinforce-tactics.git
    os.chdir(REPO_DIR)

# Install the package so all imports resolve
!pip install -q -e .

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"Working directory: {os.getcwd()}")

## 2. Imports

In [ ]:
import json
import time
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sb3_contrib import MaskablePPO

from reinforcetactics.rl.masking import make_maskable_env, make_maskable_vec_env

print("All imports successful.")

## 2b. Storage configuration

Choose where to save benchmark outputs. Set `USE_GOOGLE_DRIVE = True` to
persist results to Google Drive (recommended for Colab -- files survive
runtime disconnects). Set `False` to use the default local/Colab storage.

Each execution saves outputs under a **per-opponent, datetime-stamped subfolder**
(e.g. `benchmarks/ppo_vs_random/20250615_143022/`), so results are organized
by opponent and previous runs are preserved automatically.

In [ ]:
# --- Storage configuration ---
# Set to True to save all outputs (checkpoints, results, plots) to Google Drive.
# This is recommended when running on Google Colab, since files on the local
# Colab filesystem are lost when the runtime disconnects.
# Set to False to use the default local/Colab storage.
USE_GOOGLE_DRIVE = True

# Google Drive base directory (only used when USE_GOOGLE_DRIVE is True).
# This path is relative to your Google Drive root (MyDrive/).
# The final output path is: {DRIVE_BASE_DIR}/ppo_vs_{OPPONENT}/{datetime}/
DRIVE_BASE_DIR = "reinforce-tactics/benchmarks"

# --- Mount Google Drive if enabled ---
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        SAVE_DIR = Path("/content/drive/MyDrive") / DRIVE_BASE_DIR
        SAVE_DIR.mkdir(parents=True, exist_ok=True)
        print("Google Drive mounted successfully.")
        print(f"Save directory: {SAVE_DIR}")
    except ImportError:
        print("WARNING: google.colab not available (not running in Colab).")
        print("  Falling back to local storage.")
        USE_GOOGLE_DRIVE = False
        SAVE_DIR = None
    except Exception as e:
        print(f"WARNING: Failed to mount Google Drive: {e}")
        print("  Falling back to local storage.")
        USE_GOOGLE_DRIVE = False
        SAVE_DIR = None
else:
    SAVE_DIR = None
    print("Using local storage (default).")
    print("  Tip: Set USE_GOOGLE_DRIVE = True to persist results to Google Drive.")

## 3. Configuration

In [ ]:
# --- Benchmark settings ---
MAP_FILE = "maps/1v1/beginner.csv"  # 6x6 beginner map
OPPONENT = "random"  # Start with random for easier wins
MAX_STEPS = 400  # max env steps per episode (raised so MAX_TURNS triggers first)
MAX_TURNS = 20  # max game turns before auto-draw (terminated, not truncated)
N_ENVS = 4  # parallel training envs
SEED = 42

# Action space mode:
#   'flat_discrete'  — exact per-action masks (recommended, eliminates invalid actions)
#   'multi_discrete' — per-dimension masks (over-approximation, original behaviour)
ACTION_SPACE = "flat_discrete"

# Checkpoints to evaluate
CHECKPOINTS = [10_000, 50_000, 200_000, 1_000_000, 2_000_000]
EVAL_EPISODES = 50  # episodes per evaluation

# Opponents the trained policy is benchmarked against at every checkpoint.
# These are independent of the *training* opponent (OPPONENT) so you can
# train against one bot tier and still track win rate vs weaker / stronger
# bots over time. Supported strings: 'random', 'simple' (= SimpleBot, alias
# 'bot'), 'medium' (MediumBot), 'advanced' (AdvancedBot), 'noop'.
EVAL_OPPONENTS = ["random", "simple", "medium"]

# --- Reward configuration ---
# Tuned so capturing the enemy HQ dominates the alternative of farming
# kills against a respawning opponent. Old weights made `kill` (+10-15)
# worth more than a turn of `seize_progress` (+1), pushing the policy
# into a kill-farm local optimum that hit the step cap every episode.
REWARD_CONFIG = {
    # Terminal rewards
    "win": 1000.0,
    "loss": -1000.0,
    "draw": -200.0,
    # Potential-based shaping (small; only telescoped delta enters reward)
    "income_diff": 0.05,
    "unit_diff": 0.3,
    "structure_control": 1.0,
    # Per-action rewards — capture/seize now dominate combat farming
    "create_unit": 0.5,
    "move": 0.0,
    "damage_scale": 0.05,
    "kill": 5.0,
    "seize_progress": 5.0,
    "capture": 200.0,
    "cure": 5.0,
    "heal_scale": 0.5,
    "paralyze": 8.0,
    "haste": 6.0,
    "defence_buff": 5.0,
    "attack_buff": 5.0,
    # Penalties
    "invalid_action": -10.0,
    "turn_penalty": -2.0,
}

# PPO hyperparameters
PPO_CONFIG = dict(
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.05,  # Higher entropy for exploration
    vf_coef=0.5,
    max_grad_norm=0.5,
)

# --- Per-run output directory (opponent + datetime-stamped) ---
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_SUBDIR = f"ppo_vs_{OPPONENT}"

# Output paths — uses Google Drive if enabled, otherwise local storage.
# Each execution gets its own subfolder: ppo_vs_{OPPONENT}/{datetime}/
if SAVE_DIR is not None:
    BENCHMARK_DIR = SAVE_DIR / RUN_SUBDIR / RUN_ID
else:
    BENCHMARK_DIR = Path("benchmarks") / RUN_SUBDIR / RUN_ID
BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR = BENCHMARK_DIR / "charts"
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Map:          {MAP_FILE}")
print(f"Opponent:     {OPPONENT}")
print(f"Action space: {ACTION_SPACE}")
print(f"Max steps:    {MAX_STEPS}")
print(f"Max turns:    {MAX_TURNS}")
print(f"Checkpoints:  {CHECKPOINTS}")
print(f"Eval eps:     {EVAL_EPISODES}")
print(f"Eval bots:    {EVAL_OPPONENTS}")
print(f"ent_coef:     {PPO_CONFIG['ent_coef']}")
print(f"turn_penalty: {REWARD_CONFIG['turn_penalty']}")
print(f"draw:         {REWARD_CONFIG['draw']}")
print(f"Storage:      {'Google Drive' if USE_GOOGLE_DRIVE else 'Local'}")
print(f"Run ID:       {RUN_ID}")
print(f"Output dir:   {BENCHMARK_DIR}")

## 4. Create environments

In [ ]:
# Training envs (vectorized, headless)
vec_env = make_maskable_vec_env(
    n_envs=N_ENVS,
    map_file=MAP_FILE,
    opponent=OPPONENT,
    max_steps=MAX_STEPS,
    max_turns=MAX_TURNS,
    reward_config=REWARD_CONFIG,
    seed=SEED,
    use_subprocess=False,  # DummyVecEnv (safer in notebooks)
    action_space_type=ACTION_SPACE,
)

# One eval env per opponent we want to benchmark against. Each env is a fresh
# single-process instance so per-opponent evaluations don't share state. The
# training opponent is always included so we always have a "primary" line that
# matches the curve the agent is actually optimizing against.
_eval_opponent_list = list(dict.fromkeys([*EVAL_OPPONENTS, OPPONENT]))  # de-dupe, preserve order
eval_envs = {
    opp: make_maskable_env(
        map_file=MAP_FILE,
        opponent=opp,
        max_steps=MAX_STEPS,
        max_turns=MAX_TURNS,
        reward_config=REWARD_CONFIG,
        action_space_type=ACTION_SPACE,
        seed=SEED,
    )
    for opp in _eval_opponent_list
}

# Backward-compatible alias: eval_env is the env for the training opponent
# (used by the replay / video cells further down).
eval_env = eval_envs[OPPONENT]

print(f"Observation space: {vec_env.observation_space}")
print(f"Action space:      {vec_env.action_space}")
print(f"Eval opponents:    {list(eval_envs)}")


## 5. Create MaskablePPO model

In [ ]:
model = MaskablePPO(
    "MultiInputPolicy",
    vec_env,
    verbose=0,
    tensorboard_log=str(BENCHMARK_DIR / "tensorboard"),
    device=DEVICE,
    seed=SEED,
    **PPO_CONFIG,
)

print("MaskablePPO model created.")
print(f"Policy:  {model.policy.__class__.__name__}")
print(f"Device:  {model.device}")

## 6. Import evaluation helper

Use the shared evaluation function from the library.

In [ ]:
from reinforcetactics.rl.evaluation import evaluate_model

print("Using evaluate_model from reinforcetactics.rl.evaluation")

In [ ]:
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.utils import safe_mean


class TrainingMetricsCallback(BaseCallback):
    """Capture per-rollout PPO training metrics during ``model.learn()``.

    SB3's ``Logger.dump()`` clears ``name_to_value`` between rollouts, so
    rollout/* values are gone by the time the next callback hook fires. To
    work around this we capture rollout/* directly from
    ``self.model.ep_info_buffer`` at ``_on_rollout_end`` (the value source
    the logger itself uses) and read train/* from the logger at the *next*
    ``_on_rollout_start`` — by which point ``train()`` has run for the
    previous iteration and populated those keys. ``_on_training_end``
    picks up the final iteration that has no follow-up rollout.

    The accumulated records persist across multiple ``model.learn()``
    invocations, so the per-checkpoint training loop produces one
    continuous timeline.
    """

    TRAIN_KEYS = (
        "train/approx_kl",
        "train/clip_fraction",
        "train/entropy_loss",
        "train/explained_variance",
        "train/learning_rate",
        "train/loss",
        "train/policy_gradient_loss",
        "train/value_loss",
    )

    def __init__(self):
        super().__init__()
        self.records = []  # list of dicts: {timesteps, ...metrics}
        self._pending = None  # snapshot in progress

    def _on_step(self) -> bool:
        return True

    def _on_rollout_end(self) -> None:
        # Capture rollout stats directly from ep_info_buffer; the logger
        # clears them immediately after _dump_logs.
        snapshot = {"timesteps": self.num_timesteps}
        ep_buffer = getattr(self.model, "ep_info_buffer", None)
        if ep_buffer:
            rewards = [ep["r"] for ep in ep_buffer if "r" in ep]
            lengths = [ep["l"] for ep in ep_buffer if "l" in ep]
            if rewards:
                snapshot["rollout/ep_rew_mean"] = float(safe_mean(rewards))
            if lengths:
                snapshot["rollout/ep_len_mean"] = float(safe_mean(lengths))
        self._pending = snapshot

    def _commit_pending(self) -> None:
        """Merge train/* from the logger into the pending record and commit."""
        if self._pending is None:
            return
        for key in self.TRAIN_KEYS:
            val = self.model.logger.name_to_value.get(key)
            if val is not None:
                self._pending[key] = float(val)
        # Only emit records that contain at least one metric beyond timesteps.
        if len(self._pending) > 1:
            self.records.append(self._pending)
        self._pending = None

    def _on_rollout_start(self) -> None:
        # train() of the previous iteration has run by this hook, so
        # train/* values are now populated in the logger.
        self._commit_pending()

    def _on_training_end(self) -> None:
        self._commit_pending()


metrics_callback = TrainingMetricsCallback()
print("TrainingMetricsCallback ready.")

## 7. Train and evaluate at each checkpoint

We train incrementally: 0 → 10K → 50K → 200K → 1M → 2M timesteps,
evaluating at each checkpoint.

In [ ]:
results = []
trained_so_far = 0
start_time = time.time()

for checkpoint_idx, checkpoint_ts in enumerate(CHECKPOINTS):
    steps_to_train = checkpoint_ts - trained_so_far
    print(f"\n{'=' * 60}")
    print(f"Training {trained_so_far:,} -> {checkpoint_ts:,} ({steps_to_train:,} steps)...")
    print(f"{'=' * 60}")

    t0 = time.time()
    model.learn(
        total_timesteps=steps_to_train,
        reset_num_timesteps=False,
        progress_bar=True,
        callback=metrics_callback,
    )
    train_time = time.time() - t0
    trained_so_far = checkpoint_ts

    # Save checkpoint
    ckpt_path = BENCHMARK_DIR / f"model_{checkpoint_ts}.zip"
    model.save(str(ckpt_path))
    print(f"Saved checkpoint: {ckpt_path}")

    # Evaluate against each preselected opponent. Use a per-checkpoint seed
    # offset so eval streams are deterministic but not identical across
    # checkpoints; offset per opponent so different bots see different episode
    # streams (otherwise random vs medium would share the same draw of seeds).
    base_seed = SEED + 1000 * checkpoint_idx
    print(f"Evaluating over {EVAL_EPISODES} episodes per opponent (base seed={base_seed})...")
    per_opponent = {}
    for opp_idx, (opp_name, opp_env) in enumerate(eval_envs.items()):
        opp_seed = base_seed + 17 * opp_idx
        m = evaluate_model(model, opp_env, n_episodes=EVAL_EPISODES, seed=opp_seed)
        m["seed"] = opp_seed
        per_opponent[opp_name] = m
        print(
            f"  vs {opp_name:<10s} WR={m['win_rate'] * 100:5.1f}%  "
            f"reward={m['avg_reward']:+8.1f} (+/-{m['std_reward']:5.1f})  "
            f"len={m['avg_length']:5.1f}  W/L/D={m['wins']}/{m['losses']}/{m['draws']}"
        )

    # Primary metrics = training opponent — keeps existing plots & diagnostics
    # working unchanged, while per_opponent holds the full breakdown.
    primary = dict(per_opponent[OPPONENT])
    primary["timesteps"] = checkpoint_ts
    primary["train_time_s"] = round(train_time, 1)
    primary["per_opponent"] = per_opponent
    results.append(primary)
    print(f"  Training time:  {train_time:.1f}s")

total_time = time.time() - start_time
print(f"\nTotal wall time: {total_time / 60:.1f} minutes")
print(f"Captured {len(metrics_callback.records)} training-metric snapshots.")


## 8. Results table

In [ ]:
df = pd.DataFrame(results)
df["win_rate_pct"] = (df["win_rate"] * 100).round(1)
df["avg_reward"] = df["avg_reward"].round(1)
df["avg_length"] = df["avg_length"].round(1)

display_df = df[["timesteps", "win_rate_pct", "avg_reward", "avg_length", "wins", "losses", "draws", "train_time_s"]].copy()
display_df.columns = ["Timesteps", "Win Rate (%)", "Avg Reward", "Avg Length", "Wins", "Losses", "Draws", "Train Time (s)"]
display_df = display_df.set_index("Timesteps")
display_df

## 9. Training curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ts = [r["timesteps"] for r in results]

# Win rate
ax = axes[0]
wr = [r["win_rate"] * 100 for r in results]
ax.plot(ts, wr, "o-", color="#2196F3", linewidth=2, markersize=8)
ax.set_xlabel("Timesteps")
ax.set_ylabel("Win Rate (%)")
ax.set_title(f"Win Rate vs {OPPONENT}")
ax.set_xscale("log")
ax.set_ylim(-5, 105)
ax.axhline(y=70, color="green", linestyle="--", alpha=0.5, label="70% target")
ax.legend()
ax.grid(True, alpha=0.3)

# Average reward
ax = axes[1]
avg_r = [r["avg_reward"] for r in results]
std_r = [r["std_reward"] for r in results]
ax.plot(ts, avg_r, "o-", color="#FF9800", linewidth=2, markersize=8)
ax.fill_between(ts, [a - s for a, s in zip(avg_r, std_r)], [a + s for a, s in zip(avg_r, std_r)], alpha=0.2, color="#FF9800")
ax.set_xlabel("Timesteps")
ax.set_ylabel("Average Reward")
ax.set_title("Average Episode Reward")
ax.set_xscale("log")
ax.grid(True, alpha=0.3)

# Episode length
ax = axes[2]
avg_l = [r["avg_length"] for r in results]
std_l = [r["std_length"] for r in results]
ax.plot(ts, avg_l, "o-", color="#4CAF50", linewidth=2, markersize=8)
ax.fill_between(ts, [a - s for a, s in zip(avg_l, std_l)], [a + s for a, s in zip(avg_l, std_l)], alpha=0.2, color="#4CAF50")
ax.set_xlabel("Timesteps")
ax.set_ylabel("Average Length (steps)")
ax.set_title("Average Episode Length")
ax.set_xscale("log")
ax.grid(True, alpha=0.3)

fig.suptitle(f"PPO Baseline Benchmarks  |  6x6 beginner map  |  vs {OPPONENT}", fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()

fig.savefig(str(CHARTS_DIR / "training_curves.png"), dpi=150, bbox_inches="tight")
print(f"Saved plot: {CHARTS_DIR / 'training_curves.png'}")
plt.show()

## 9aa. Per-opponent win-rate curves

Win rate, reward and episode length plotted per *evaluation opponent*.
This is the multi-opponent generalization of section 9 — useful for
spotting cases where the agent overfits to its training opponent and
stops generalizing to weaker / stronger bots.


In [ ]:
# Per-opponent benchmark curves. Falls back gracefully if the training loop
# was run before this notebook gained multi-opponent support (older runs only
# have flat metrics, no "per_opponent" key).
if not results or "per_opponent" not in results[0]:
    print("No per-opponent results to plot — re-run section 7 (training loop).")
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    timesteps_axis = [r["timesteps"] for r in results]
    opponent_names = list(results[-1]["per_opponent"].keys())

    cmap = plt.get_cmap("tab10")
    colors = {opp: cmap(i % 10) for i, opp in enumerate(opponent_names)}

    # Win rate per opponent
    ax = axes[0]
    for opp in opponent_names:
        wr = [r["per_opponent"][opp]["win_rate"] * 100 for r in results if opp in r["per_opponent"]]
        ts = [r["timesteps"] for r in results if opp in r["per_opponent"]]
        ax.plot(ts, wr, "o-", label=opp, color=colors[opp], linewidth=2, markersize=6)
    ax.set_xlabel("Timesteps")
    ax.set_ylabel("Win Rate (%)")
    ax.set_title("Win rate per opponent")
    ax.set_xscale("log")
    ax.set_ylim(-5, 105)
    ax.axhline(y=70, color="green", linestyle="--", alpha=0.4, label="70% target")
    ax.legend(fontsize=8, loc="best")
    ax.grid(True, alpha=0.3)

    # Average reward per opponent
    ax = axes[1]
    for opp in opponent_names:
        avg_r = [r["per_opponent"][opp]["avg_reward"] for r in results if opp in r["per_opponent"]]
        ts = [r["timesteps"] for r in results if opp in r["per_opponent"]]
        ax.plot(ts, avg_r, "o-", label=opp, color=colors[opp], linewidth=2, markersize=6)
    ax.set_xlabel("Timesteps")
    ax.set_ylabel("Average reward")
    ax.set_title("Average reward per opponent")
    ax.set_xscale("log")
    ax.legend(fontsize=8, loc="best")
    ax.grid(True, alpha=0.3)

    # Episode length per opponent
    ax = axes[2]
    for opp in opponent_names:
        avg_l = [r["per_opponent"][opp]["avg_length"] for r in results if opp in r["per_opponent"]]
        ts = [r["timesteps"] for r in results if opp in r["per_opponent"]]
        ax.plot(ts, avg_l, "o-", label=opp, color=colors[opp], linewidth=2, markersize=6)
    ax.set_xlabel("Timesteps")
    ax.set_ylabel("Average length (steps)")
    ax.set_title("Average episode length per opponent")
    ax.set_xscale("log")
    ax.legend(fontsize=8, loc="best")
    ax.grid(True, alpha=0.3)

    fig.suptitle(
        f"Per-opponent benchmarks  |  trained vs {OPPONENT}",
        fontsize=13,
        fontweight="bold",
        y=1.02,
    )
    fig.tight_layout()
    plot_path = CHARTS_DIR / "per_opponent_curves.png"
    fig.savefig(str(plot_path), dpi=150, bbox_inches="tight")
    print(f"Saved: {plot_path}")
    plt.show()

    # Persist a flat per-opponent results table for diffing across runs.
    rows = []
    for r in results:
        for opp, m in r["per_opponent"].items():
            rows.append({
                "timesteps": r["timesteps"],
                "opponent": opp,
                "win_rate": m["win_rate"],
                "avg_reward": m["avg_reward"],
                "std_reward": m["std_reward"],
                "avg_length": m["avg_length"],
                "wins": m["wins"],
                "losses": m["losses"],
                "draws": m["draws"],
            })
    per_opp_df = pd.DataFrame(rows)
    per_opp_csv = BENCHMARK_DIR / "per_opponent_results.csv"
    per_opp_df.to_csv(per_opp_csv, index=False)
    print(f"Saved: {per_opp_csv}  ({len(per_opp_df)} rows)")


## 9a. Per-checkpoint distributions

Mean ± std hides important detail. The plots below expand the same
checkpoint data into:

- **Episode reward distribution** — box plot per checkpoint. Bimodal
  distributions (e.g. some big wins + many bad draws averaging out)
  are invisible in the line plot above but obvious here.
- **Episode length distribution** — same idea for `avg_length`. A flat
  median means episodes consistently hit the step limit; spread tells
  you whether the agent sometimes wins quickly.
- **Outcome breakdown (W/L/D stacked bar)** — at-a-glance view of
  whether wins are climbing, losses are growing, or the agent is
  trading wins for draws.

In [ ]:
if not results:
    print("No results to plot — run the training loop first.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("Per-checkpoint distributions", fontsize=13, fontweight="bold", y=1.02)

    timesteps = [r["timesteps"] for r in results]
    labels = [f"{ts:,}" for ts in timesteps]

    # --- 1. Episode reward distribution ---
    ax = axes[0]
    rewards_lists = [r.get("rewards", []) for r in results]
    if any(rewards_lists):
        bp = ax.boxplot(rewards_lists, labels=labels, patch_artist=True, showmeans=True)
        for patch in bp["boxes"]:
            patch.set_facecolor("#FF9800")
            patch.set_alpha(0.5)
    ax.set_xlabel("Timesteps")
    ax.set_ylabel("Episode Reward")
    ax.set_title("Episode Reward Distribution")
    ax.tick_params(axis="x", rotation=30)
    ax.grid(True, alpha=0.3, axis="y")

    # --- 2. Episode length distribution ---
    ax = axes[1]
    lengths_lists = [r.get("lengths", []) for r in results]
    if any(lengths_lists):
        bp = ax.boxplot(lengths_lists, labels=labels, patch_artist=True, showmeans=True)
        for patch in bp["boxes"]:
            patch.set_facecolor("#4CAF50")
            patch.set_alpha(0.5)
    ax.set_xlabel("Timesteps")
    ax.set_ylabel("Episode Length (steps)")
    ax.set_title("Episode Length Distribution")
    ax.tick_params(axis="x", rotation=30)
    ax.grid(True, alpha=0.3, axis="y")

    # --- 3. W/L/D stacked bar ---
    ax = axes[2]
    wins = [r["wins"] for r in results]
    losses = [r["losses"] for r in results]
    draws = [r["draws"] for r in results]
    x = np.arange(len(timesteps))
    ax.bar(x, wins, label="Wins", color="#4CAF50", edgecolor="white", linewidth=0.5)
    ax.bar(x, losses, bottom=wins, label="Losses", color="#F44336", edgecolor="white", linewidth=0.5)
    ax.bar(
        x,
        draws,
        bottom=[w + loss_count for w, loss_count in zip(wins, losses)],
        label="Draws",
        color="#FF9800",
        edgecolor="white",
        linewidth=0.5,
    )
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30)
    ax.set_xlabel("Timesteps")
    ax.set_ylabel("Episode count")
    ax.set_title(f"Outcomes per checkpoint (n={EVAL_EPISODES})")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis="y")

    fig.tight_layout()
    plot_path = CHARTS_DIR / "per_checkpoint_distributions.png"
    fig.savefig(str(plot_path), dpi=150, bbox_inches="tight")
    print(f"Saved: {plot_path}")
    plt.show()

## 9b. PPO training internals

The training-curve plot above shows what the agent achieves at each
checkpoint, but **why** the agent is (or isn't) learning is hidden inside
the PPO update loop. The cell below plots the per-rollout metrics
captured by `TrainingMetricsCallback` so you can spot common failure
modes at a glance:

| Metric | What to look for |
|--------|------------------|
| `rollout/ep_rew_mean` | Continuous learning curve (much denser than the 5-checkpoint plot) |
| `rollout/ep_len_mean` | Episodes shortening as the agent learns to finish games |
| `train/approx_kl` | Healthy: < 0.02. Above 0.05 → policy updating too aggressively (lr too high) |
| `train/clip_fraction` | Healthy: < 0.3. Near 1 → clipping firing constantly, instability |
| `train/explained_variance` | Target → 1. Stuck near 0 → value head isn't learning, advantages are noise |
| `train/entropy_loss` | Less negative = less exploration. Crashing fast → premature commitment |
| `train/policy_gradient_loss` | Should oscillate around 0; growing magnitude can indicate divergence |
| `train/value_loss` | Should decrease as the value head fits returns |

In [ ]:
if not metrics_callback.records:
    print("No training metrics captured — run the training loop first.")
else:
    df_metrics = pd.DataFrame(metrics_callback.records)
    ts_axis = df_metrics["timesteps"]

    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    fig.suptitle(
        "PPO training internals (per rollout)",
        fontsize=13,
        fontweight="bold",
        y=1.02,
    )

    def _plot(ax, col, title, *, hline_target=None, hline_danger=None):
        if col not in df_metrics.columns:
            ax.set_title(f"{title}\n(not logged)")
            ax.text(
                0.5,
                0.5,
                "no data",
                ha="center",
                va="center",
                color="#999",
                transform=ax.transAxes,
            )
            ax.set_xticks([])
            ax.set_yticks([])
            return
        ax.plot(ts_axis, df_metrics[col], color="#2196F3", linewidth=1.5)
        ax.set_xlabel("Timesteps")
        ax.set_title(title)
        ax.grid(True, alpha=0.3)
        if hline_target is not None:
            ax.axhline(
                hline_target,
                color="#4CAF50",
                linestyle="--",
                alpha=0.6,
                linewidth=1,
                label=f"target ({hline_target})",
            )
        if hline_danger is not None:
            ax.axhline(
                hline_danger,
                color="#F44336",
                linestyle="--",
                alpha=0.6,
                linewidth=1,
                label=f"danger ({hline_danger})",
            )
        if hline_target is not None or hline_danger is not None:
            ax.legend(fontsize=7, loc="best")

    _plot(axes[0, 0], "rollout/ep_rew_mean", "Episode reward (mean)")
    _plot(axes[0, 1], "rollout/ep_len_mean", "Episode length (mean)")
    _plot(
        axes[0, 2],
        "train/approx_kl",
        "approx_kl",
        hline_target=0.02,
        hline_danger=0.05,
    )
    _plot(
        axes[0, 3],
        "train/clip_fraction",
        "clip_fraction",
        hline_target=0.3,
        hline_danger=0.5,
    )
    _plot(
        axes[1, 0],
        "train/explained_variance",
        "explained_variance",
        hline_target=1.0,
        hline_danger=0.1,
    )
    _plot(axes[1, 1], "train/entropy_loss", "entropy_loss")
    _plot(axes[1, 2], "train/policy_gradient_loss", "policy_gradient_loss")
    _plot(axes[1, 3], "train/value_loss", "value_loss")

    fig.tight_layout()
    plot_path = CHARTS_DIR / "ppo_internals.png"
    fig.savefig(str(plot_path), dpi=150, bbox_inches="tight")
    print(f"Saved: {plot_path}")
    plt.show()

    # Persist the raw metrics so they can be diffed across runs.
    metrics_csv = BENCHMARK_DIR / "training_metrics.csv"
    df_metrics.to_csv(metrics_csv, index=False)
    print(f"Saved: {metrics_csv}  ({len(df_metrics)} rows)")

## 9c. Diagnose training failures

If you see **0% win rate** and all games ending as **draws at 500 steps**, the agent
is not learning to win — it's farming shaping rewards. Run the cell below to
diagnose the specific failure mode and get targeted recommendations.

In [ ]:
def diagnose_training(results):
    """Analyze benchmark results and print a diagnosis with relative recommendations."""
    if not results:
        print("No results to analyze.")
        return

    # --- Collect signals ---
    win_rates = [r["win_rate"] for r in results]
    avg_rewards = [r["avg_reward"] for r in results]
    avg_lengths = [r["avg_length"] for r in results]
    draw_counts = [r["draws"] for r in results]
    loss_counts = [r["losses"] for r in results]
    n_eval = results[0].get("wins", 0) + results[0].get("losses", 0) + results[0].get("draws", 0)

    all_draws = all(d == n_eval for d in draw_counts)
    all_max_len = all(abs(length - MAX_STEPS) < 1.0 for length in avg_lengths)
    rewards_positive = all(r > 0 for r in avg_rewards)
    rewards_declining = len(avg_rewards) >= 2 and avg_rewards[-1] < avg_rewards[0] * 0.7
    no_wins = all(w == 0.0 for w in win_rates)
    late_losses = loss_counts[-1] > 0 and all(lc == 0 for lc in loss_counts[:-1])
    all_losses_late = loss_counts[-1] == n_eval if len(loss_counts) > 0 else False
    negative_rewards = all(r < -1000 for r in avg_rewards[:-1]) if len(avg_rewards) > 1 else False

    # Current config snapshot (used by recommendation helpers below)
    rc = REWARD_CONFIG
    pc = PPO_CONFIG

    print("=" * 65)
    print("  TRAINING DIAGNOSTICS")
    print("=" * 65)

    # --- Summary table ---
    print(f"\n{'Timesteps':>12}  {'WR':>6}  {'Reward':>10}  {'Length':>8}  {'W/L/D'}")
    print("-" * 58)
    for r in results:
        ts = r["timesteps"]
        wr = r["win_rate"] * 100
        rw = r["avg_reward"]
        ln = r["avg_length"]
        wld = f"{r['wins']}/{r['losses']}/{r['draws']}"
        print(f"{ts:>12,}  {wr:>5.1f}%  {rw:>10.1f}  {ln:>8.1f}  {wld}")

    # --- Failure mode detection ---
    print(f"\n{'─' * 65}")
    print("  DIAGNOSIS")
    print(f"{'─' * 65}")

    if no_wins and all_draws and all_max_len and rewards_positive:
        print(f"""
FAILURE MODE: Shaping-reward stalemate

The agent takes only valid actions, earns positive shaping rewards by
creating units and controlling structures, but never finishes a game
within the {MAX_STEPS}-step limit.

Root cause: The agent has NEVER experienced a +{rc["win"]:.0f} win or {rc["loss"]:.0f} loss
reward. With no terminal signal, it optimizes shaping rewards instead
of pursuing victory.
""")
        if rewards_declining:
            print(f"⚠  Reward dropped from {avg_rewards[0]:.0f} → {avg_rewards[-1]:.0f}")
            print("   suggesting policy degradation over time.\n")

    elif no_wins and negative_rewards and all_max_len:
        print(f"""
FAILURE MODE: Invalid-action penalty flood (multi_discrete)

~99% of sampled actions are game-invalid due to per-dimension mask
over-approximation. The {rc.get("invalid_action", -10)} penalty per invalid action dominates the
reward signal, making it impossible to learn from actual gameplay.
""")
        if all_losses_late:
            print(
                "At the final checkpoint the agent collapsed to spamming end_turn\n"
                "to avoid penalties, letting the opponent win every game.\n"
            )

    elif no_wins and late_losses:
        print("""
FAILURE MODE: Policy collapse

Early episodes stalemate (draws), then the agent collapses to a
degenerate strategy (e.g., always ending turn) and starts losing.
""")
    else:
        if max(win_rates) > 0:
            best_idx = win_rates.index(max(win_rates))
            print(f"\nBest win rate: {max(win_rates) * 100:.1f}% at {results[best_idx]['timesteps']:,} timesteps.")
            if win_rates[-1] < max(win_rates):
                print("Win rate is declining — possible overfitting or learning rate too high.")
        else:
            print("\n0% win rate across all checkpoints. Review environment configuration.")

    # --- Should I train longer? ---
    print(f"{'─' * 65}")
    print("  SHOULD YOU TRAIN LONGER?")
    print(f"{'─' * 65}")

    if no_wins and all_draws:
        print("""
  NO. Training longer will not help.

  The agent is stuck in a local optimum (maximizing shaping rewards
  without learning to win). More timesteps will not change this —
  the agent needs a different reward structure to break out.
""")
    elif no_wins:
        print("""
  NO. The current configuration has a fundamental issue preventing
  the agent from learning to win. Fix the issues below first.
""")
    elif win_rates[-1] > win_rates[-2] if len(win_rates) >= 2 else False:
        print("""
  MAYBE. Win rate is still increasing — more training could help.
  But check if the rate of improvement is slowing significantly.
""")
    else:
        print("""
  PROBABLY NOT. Win rate has plateaued or is declining.
  Consider tuning hyperparameters or reward configuration.
""")

    # --- Recommendations (relative to current values; skipped if already in range) ---
    print(f"{'─' * 65}")
    print("  RECOMMENDED FIXES (relative to current config)")
    print(f"{'─' * 65}")

    # Each helper returns either (title, detail) or None when the current
    # value is already at or beyond the recommended target.

    def _halve_steps():
        cur = MAX_STEPS
        if cur <= 100:
            return None  # Already short — making it shorter risks losing depth
        target = max(50, cur // 2)
        return (
            "Reduce MAX_STEPS",
            f"  Current: {cur}. Try {target} (≈ half).\n"
            "  Shorter episodes force earlier confrontation and make terminal\n"
            "  rewards reachable. Edit MAX_STEPS in the config cell above.",
        )

    def _more_negative_draw():
        cur = float(rc.get("draw", 0.0))
        if cur <= -1000:
            return None  # Already on par with a loss
        target = round(cur * 1.5, 1) if cur < 0 else -200.0
        return (
            "Make truncation penalty more negative",
            f"  Current REWARD_CONFIG['draw'] = {cur}. Try {target} (≈ 1.5× more negative).\n"
            "  When episodes time out, a stronger negative signal pushes the\n"
            "  agent to finish games rather than stall.",
        )

    def _shrink_shaping():
        items = []
        for key in ("structure_control", "unit_diff", "income_diff"):
            cur = float(rc.get(key, 0.0))
            if cur <= 0.001:
                continue  # Already negligible
            items.append(f"{key}: {cur} → {round(cur / 2, 3)}")
        if not items:
            return None
        body = "\n  ".join(items)
        return (
            "Halve shaping reward magnitudes",
            f"  Current values:\n  {body}\n"
            "  Smaller shaping rewards reduce the incentive to farm them\n"
            "  rather than pursue victory.",
        )

    def _more_negative_turn_penalty():
        cur = float(rc.get("turn_penalty", 0.0))
        if cur <= -10:
            return None  # Already heavily punitive
        target = round(cur * 1.5, 1) if cur < 0 else -2.0
        return (
            "Make turn_penalty more negative",
            f"  Current REWARD_CONFIG['turn_penalty'] = {cur}. Try {target} (≈ 1.5× more negative).\n"
            "  Stronger end-turn penalty discourages stalling without\n"
            "  preventing decisive play.",
        )

    def _bump_ent_coef():
        cur = float(pc.get("ent_coef", 0.0))
        target = round(min(cur + 0.02, 0.1), 3)
        if target <= cur:
            return None  # Already at or above the recommended cap
        return (
            "Increase ent_coef for more exploration",
            f"  Current PPO_CONFIG['ent_coef'] = {cur}. Try {target} (cap 0.1).\n"
            "  Higher entropy keeps the agent exploring aggressive\n"
            "  strategies instead of narrowing too quickly.",
        )

    def _weaker_opponent():
        if OPPONENT == "random":
            return None  # Already on the weakest opponent
        return (
            "Start with a weaker opponent",
            f"  Current: OPPONENT='{OPPONENT}'. Try 'random' first so the agent\n"
            "  experiences wins early, then graduate back to 'bot'.\n"
            "  Edit OPPONENT in the config cell above.",
        )

    def _switch_action_space():
        if ACTION_SPACE == "flat_discrete":
            return None  # Already on the recommended action space
        return (
            "Switch to flat_discrete action space",
            "  Eliminates ~99% invalid actions from per-dimension masking.\n"
            "  Set ACTION_SPACE = 'flat_discrete' in the config cell.",
        )

    def _shrink_invalid_penalty():
        cur = float(rc.get("invalid_action", 0.0))
        if cur >= -0.5:
            return None  # Already small in magnitude
        target = round(cur / 2, 2)
        return (
            "Halve invalid_action penalty magnitude",
            f"  Current REWARD_CONFIG['invalid_action'] = {cur}. Try {target} (half magnitude).\n"
            "  Useful with multi_discrete; less critical once flat_discrete is on.",
        )

    if no_wins and all_draws and all_max_len:
        candidates = [
            _halve_steps(),
            _more_negative_draw(),
            _shrink_shaping(),
            _more_negative_turn_penalty(),
            _bump_ent_coef(),
            _weaker_opponent(),
        ]
    elif negative_rewards:
        candidates = [
            _switch_action_space(),
            _shrink_invalid_penalty(),
        ]
    else:
        candidates = []

    fixes = [c for c in candidates if c is not None]

    if not fixes:
        print(
            "\n  No targeted fixes — your reward / PPO / opponent settings are already at\n"
            "  or near the recommended values. If training is still failing, consider:\n"
            "  - Increasing total timesteps\n"
            "  - Reviewing the environment for bugs (action masking, opponent dispatch)\n"
            "  - Trying a different random seed"
        )
    else:
        for i, (title, detail) in enumerate(fixes, 1):
            print(f"\n  {i}. {title}\n{detail}")

    print(f"\n{'=' * 65}")


diagnose_training(results)

## 9d. Evaluation replay log

Record every move the agent makes during evaluation and save to JSON.
This lets you inspect exactly what the agent is doing each step — is it
spamming end_turn? Never attacking? Ignoring enemies?

Set `N_REPLAY_EPISODES` to control how many episodes to record
(default 3 to keep file sizes small).

In [ ]:
N_REPLAY_EPISODES = 3  # episodes to record per checkpoint

ACTION_NAMES = [
    "create_unit",
    "move",
    "attack",
    "seize",
    "heal",
    "end_turn",
    "paralyze",
    "haste",
    "defence_buff",
    "attack_buff",
]
UNIT_NAMES = ["W", "M", "C", "A", "K", "R", "S", "B"]


def _snapshot_game_state(env):
    """Capture a summary of the current game state."""
    gs = env.unwrapped.game_state
    ap = env.unwrapped.agent_player
    opp = 3 - ap
    return {
        "agent_gold": gs.player_gold.get(ap, 0),
        "opponent_gold": gs.player_gold.get(opp, 0),
        "agent_units": sum(1 for u in gs.units if u.player == ap),
        "opponent_units": sum(1 for u in gs.units if u.player == opp),
        "agent_structures": len(gs.grid.get_capturable_tiles(player=ap)),
        "opponent_structures": len(gs.grid.get_capturable_tiles(player=opp)),
        "turn": gs.turn_number,
    }


def evaluate_with_replay(model, env, n_episodes=3):
    """
    Run evaluation episodes and record every action to a replay log.

    Returns:
        List of episode dicts, each containing 'steps', 'outcome', etc.
    """
    # Resolve which player the agent controls once — the env's agent_player
    # never changes during evaluation, but it can be 2 (e.g. self-play envs)
    # so we must not hardcode it.
    agent_player = env.unwrapped.agent_player

    episodes = []

    for ep_idx in range(n_episodes):
        obs, _ = env.reset()
        done = False
        ep_reward = 0.0
        steps = []
        step_num = 0

        while not done:
            masks = env.action_masks()
            action, _ = model.predict(obs, deterministic=True, action_masks=masks)

            # Decode the action BEFORE stepping
            raw_action = action
            if ACTION_SPACE == "flat_discrete":
                action_idx = int(action)
                inner = env.unwrapped
                if 0 <= action_idx < len(inner._current_actions):
                    action_arr = inner._current_actions[action_idx]
                else:
                    action_arr = np.array([5, 0, 0, 0, 0, 0])
            else:
                action_arr = np.asarray(action)

            action_type = int(action_arr[0])
            action_name = ACTION_NAMES[action_type] if action_type < len(ACTION_NAMES) else f"unknown_{action_type}"
            unit_type = UNIT_NAMES[int(action_arr[1]) % 8]
            from_pos = [int(action_arr[2]), int(action_arr[3])]
            to_pos = [int(action_arr[4]), int(action_arr[5])]

            # Step
            obs, reward, terminated, truncated, info = env.step(raw_action)
            ep_reward += reward
            step_num += 1
            done = terminated or truncated

            # Per-step diagnostics from info: reward_breakdown lets us
            # decompose total reward by source (action / shaping / terminal /
            # invalid penalty); n_legal_actions tracks mask coverage so we
            # can see whether gradients are diluted across mostly-dead
            # output neurons.
            breakdown = info.get("reward_breakdown", {})
            step_record = {
                "step": step_num,
                "action": action_name,
                "unit_type": unit_type if action_type == 0 else None,
                "from": from_pos,
                "to": to_pos,
                "reward": round(float(reward), 3),
                "cumulative_reward": round(float(ep_reward), 3),
                "valid": info.get("valid_action", True),
                "reward_breakdown": {k: round(float(v), 3) for k, v in breakdown.items()},
                "n_legal_actions": int(info.get("n_legal_actions", 0)),
                "game_state": _snapshot_game_state(env),
            }
            steps.append(step_record)

        winner = info.get("winner")
        if winner == agent_player:
            outcome = "win"
        elif winner is not None:
            outcome = "loss"
        else:
            outcome = "draw"

        # Extract final turn count from last step's game state
        final_turn = steps[-1]["game_state"]["turn"] if steps else 0

        episodes.append(
            {
                "episode": ep_idx,
                "outcome": outcome,
                "total_reward": round(float(ep_reward), 2),
                "length": step_num,
                "turns": final_turn,
                "steps": steps,
            }
        )

    return episodes


print("evaluate_with_replay() defined.")

In [ ]:
# Record replays from the final checkpoint
print(f"Recording {N_REPLAY_EPISODES} replay episodes from final model...\n")
replay_episodes = evaluate_with_replay(model, eval_env, n_episodes=N_REPLAY_EPISODES)

# Save to JSON (includes full reward & PPO config for reproducibility)
replay_path = BENCHMARK_DIR / "eval_replays.json"
replay_data = {
    "metadata": {
        "timesteps": CHECKPOINTS[-1],
        "map": MAP_FILE,
        "opponent": OPPONENT,
        "action_space": ACTION_SPACE,
        "max_steps": MAX_STEPS,
        "reward_config": REWARD_CONFIG,
        "ppo_config": PPO_CONFIG,
    },
    "episodes": replay_episodes,
}
with open(replay_path, "w") as f:
    json.dump(replay_data, f, indent=2)
print(f"Saved replays: {replay_path}")

# Save reward config as standalone file for easy diffing between runs
reward_config_path = BENCHMARK_DIR / "reward_config.json"
reward_config_data = {
    "date": datetime.now().isoformat(),
    "reward_config": REWARD_CONFIG,
    "ppo_config": PPO_CONFIG,
}
with open(reward_config_path, "w") as f:
    json.dump(reward_config_data, f, indent=2)
print(f"Saved config:  {reward_config_path}")

# --- Print summary for each episode ---
for ep in replay_episodes:
    # Extract turn count from the last step's game state
    final_turn = ep["steps"][-1]["game_state"]["turn"] if ep["steps"] else 0

    print(f"\n{'─' * 55}")
    print(
        f"Episode {ep['episode']}  |  outcome={ep['outcome']}  |  "
        f"length={ep['length']}  |  turns={final_turn}  |  "
        f"reward={ep['total_reward']}"
    )
    print(f"{'─' * 55}")

    # Action distribution
    from collections import Counter

    action_counts = Counter(s["action"] for s in ep["steps"])
    total = len(ep["steps"])
    print(f"\n  Action distribution ({total} steps):")
    for action, count in action_counts.most_common():
        pct = count / total * 100
        bar = "#" * int(pct / 2)
        print(f"    {action:15s}  {count:4d}  ({pct:5.1f}%)  {bar}")

    # Unit creation breakdown (by unit type)
    create_steps = [s for s in ep["steps"] if s["action"] == "create_unit"]
    if create_steps:
        unit_counts = Counter(s["unit_type"] for s in create_steps)
        print(f"\n  Units created ({len(create_steps)} total):")
        for ut, count in unit_counts.most_common():
            bar = "#" * count
            print(f"    {ut:3s}  {count:3d}  {bar}")

    # Captures: detect when agent_structures increases after a seize action
    captures = 0
    for i, s in enumerate(ep["steps"]):
        if s["action"] == "seize" and s["valid"] and i > 0:
            prev_structures = ep["steps"][i - 1]["game_state"]["agent_structures"]
            curr_structures = s["game_state"]["agent_structures"]
            if curr_structures > prev_structures:
                captures += 1
    seize_count = action_counts.get("seize", 0)
    print(f"\n  Structures captured: {captures} (from {seize_count} seize actions)")

    # Per-turn breakdown: how many steps (actions) the agent took each turn
    turns = {}
    for s in ep["steps"]:
        t = s["game_state"]["turn"]
        turns.setdefault(t, 0)
        turns[t] += 1
    print(f"\n  Steps per turn ({len(turns)} turns):")
    for t in sorted(turns.keys()):
        bar = "#" * min(turns[t], 40)
        print(f"    turn {t:3d}: {turns[t]:3d} steps  {bar}")
    if final_turn > 0:
        print(f"    avg: {total / final_turn:.1f} steps/turn")

    # Invalid action count
    invalid = sum(1 for s in ep["steps"] if not s["valid"])
    if invalid > 0:
        print(f"\n  Invalid actions: {invalid}/{total} ({invalid / total * 100:.1f}%)")

    # Game state at end
    final = ep["steps"][-1]["game_state"]
    print(f"\n  Final state (step {ep['length']}, turn {final['turn']}):")
    print(f"    Agent:    {final['agent_units']} units, {final['agent_structures']} structures, {final['agent_gold']} gold")
    print(
        f"    Opponent: {final['opponent_units']} units, "
        f"{final['opponent_structures']} structures, "
        f"{final['opponent_gold']} gold"
    )

    # Show first 10 and last 10 moves
    steps = ep["steps"]
    print("\n  First 10 moves:")
    for s in steps[:10]:
        gs = s["game_state"]
        ut = f" ({s['unit_type']})" if s["unit_type"] else ""
        valid_marker = "" if s["valid"] else " [INVALID]"
        print(
            f"    step {s['step']:3d}  {s['action']:15s}{ut:5s}  "
            f"{s['from']}→{s['to']}  r={s['reward']:+.1f}  "
            f"units={gs['agent_units']}v{gs['opponent_units']}  "
            f"turn={gs['turn']}{valid_marker}"
        )

    if len(steps) > 20:
        print(f"    ... ({len(steps) - 20} steps omitted) ...")

    if len(steps) > 10:
        print("  Last 10 moves:")
        for s in steps[-10:]:
            gs = s["game_state"]
            ut = f" ({s['unit_type']})" if s["unit_type"] else ""
            valid_marker = "" if s["valid"] else " [INVALID]"
            print(
                f"    step {s['step']:3d}  {s['action']:15s}{ut:5s}  "
                f"{s['from']}→{s['to']}  r={s['reward']:+.1f}  "
                f"units={gs['agent_units']}v{gs['opponent_units']}  "
                f"turn={gs['turn']}{valid_marker}"
            )

## 9e. Replay visualizations

Charts that make it easier to spot behavioural patterns across episodes:

- **Action distribution** — what the agent spends its time doing
- **Unit & gold progression** — economy and army over turns
- **Cumulative reward** — where reward comes from (or stalls)
- **Steps per turn** — tactical complexity / end-turn spam detection
- **Cross-episode comparison** — side-by-side summary

In [ ]:
from collections import Counter

import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt

# --- Colour palette ---
ACTION_COLORS = {
    "move": "#4CAF50",
    "attack": "#F44336",
    "end_turn": "#9E9E9E",
    "create_unit": "#2196F3",
    "seize": "#FF9800",
    "heal": "#E91E63",
    "paralyze": "#9C27B0",
    "haste": "#00BCD4",
    "defence_buff": "#795548",
    "attack_buff": "#FF5722",
}
AGENT_COLOR = "#2196F3"
OPP_COLOR = "#F44336"

ORDERED_ACTIONS = [
    "move",
    "attack",
    "create_unit",
    "end_turn",
    "seize",
    "heal",
    "paralyze",
    "haste",
    "defence_buff",
    "attack_buff",
]


def _per_turn(steps, key):
    """Aggregate a game-state key per turn (take value at end of turn)."""
    turn_vals = {}
    for s in steps:
        t = s["game_state"]["turn"]
        turn_vals[t] = s["game_state"][key]
    turns = sorted(turn_vals)
    return turns, [turn_vals[t] for t in turns]


def plot_episode(ep, ep_idx=None):
    """Create a 2x3 dashboard for a single replay episode."""
    steps = ep["steps"]
    idx = ep_idx if ep_idx is not None else ep.get("episode", "?")
    outcome = ep["outcome"].upper()
    total_reward = ep["total_reward"]
    length = ep["length"]
    final_turn = steps[-1]["game_state"]["turn"] if steps else 0

    fig = plt.figure(figsize=(16, 9))
    gs = gridspec.GridSpec(2, 3, hspace=0.35, wspace=0.35)

    outcome_color = {"WIN": "#4CAF50", "LOSS": "#F44336", "DRAW": "#FF9800"}
    fig.suptitle(
        f"Episode {idx}  —  {outcome}  |  {length} steps  |  {final_turn} turns  |  reward {total_reward:+.0f}",
        fontsize=14,
        fontweight="bold",
        color=outcome_color.get(outcome, "black"),
    )

    # --- 1. Action distribution (horizontal bar) ---
    ax = fig.add_subplot(gs[0, 0])
    action_counts = Counter(s["action"] for s in steps)
    actions = [a for a in ORDERED_ACTIONS if action_counts.get(a, 0) > 0]
    counts = [action_counts[a] for a in actions]
    colors = [ACTION_COLORS.get(a, "#607D8B") for a in actions]
    bars = ax.barh(actions, counts, color=colors, edgecolor="white", linewidth=0.5)
    ax.set_xlabel("Count")
    ax.set_title("Action Distribution")
    ax.invert_yaxis()
    for bar, c in zip(bars, counts):
        pct = c / len(steps) * 100
        ax.text(
            bar.get_width() + max(counts) * 0.02,
            bar.get_y() + bar.get_height() / 2,
            f"{pct:.0f}%",
            va="center",
            fontsize=9,
            color="#555",
        )

    # --- 2. Unit count over turns ---
    ax = fig.add_subplot(gs[0, 1])
    turns_a, agent_units = _per_turn(steps, "agent_units")
    _, opp_units = _per_turn(steps, "opponent_units")
    ax.plot(turns_a, agent_units, color=AGENT_COLOR, linewidth=1.5, label="Agent")
    ax.plot(turns_a, opp_units, color=OPP_COLOR, linewidth=1.5, label="Opponent")
    ax.fill_between(turns_a, agent_units, alpha=0.1, color=AGENT_COLOR)
    ax.fill_between(turns_a, opp_units, alpha=0.1, color=OPP_COLOR)
    ax.set_xlabel("Turn")
    ax.set_ylabel("Units")
    ax.set_title("Army Size")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

    # --- 3. Gold over turns ---
    ax = fig.add_subplot(gs[0, 2])
    _, agent_gold = _per_turn(steps, "agent_gold")
    _, opp_gold = _per_turn(steps, "opponent_gold")
    ax.plot(turns_a, agent_gold, color=AGENT_COLOR, linewidth=1.5, label="Agent")
    ax.plot(turns_a, opp_gold, color=OPP_COLOR, linewidth=1.5, label="Opponent")
    ax.set_xlabel("Turn")
    ax.set_ylabel("Gold")
    ax.set_title("Economy")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

    # --- 4. Cumulative reward ---
    ax = fig.add_subplot(gs[1, 0])
    cum_r = [s["cumulative_reward"] for s in steps]
    step_nums = [s["step"] for s in steps]
    ax.plot(step_nums, cum_r, color="#FF9800", linewidth=1.2)
    ax.axhline(0, color="grey", linewidth=0.5, linestyle="--")
    ax.set_xlabel("Step")
    ax.set_ylabel("Cumulative Reward")
    ax.set_title("Reward Curve")
    ax.grid(True, alpha=0.2)

    # --- 5. Steps per turn ---
    ax = fig.add_subplot(gs[1, 1])
    turn_steps = {}
    for s in steps:
        t = s["game_state"]["turn"]
        turn_steps[t] = turn_steps.get(t, 0) + 1
    ts_sorted = sorted(turn_steps)
    spt = [turn_steps[t] for t in ts_sorted]
    # Colour end-turn-only turns (1 step = just end_turn)
    bar_colors = ["#F44336" if v == 1 else "#4CAF50" for v in spt]
    ax.bar(ts_sorted, spt, color=bar_colors, width=1.0, edgecolor="white", linewidth=0.3)
    avg_spt = np.mean(spt)
    ax.axhline(avg_spt, color="#2196F3", linewidth=1, linestyle="--", label=f"avg {avg_spt:.1f}")
    ax.set_xlabel("Turn")
    ax.set_ylabel("Steps")
    ax.set_title("Steps per Turn")
    ax.legend(fontsize=8)

    # --- 6. Unit creation breakdown ---
    ax = fig.add_subplot(gs[1, 2])
    create_steps = [s for s in steps if s["action"] == "create_unit"]
    if create_steps:
        unit_counts = Counter(s["unit_type"] for s in create_steps)
        labels = list(unit_counts.keys())
        sizes = list(unit_counts.values())
        wedge_colors = plt.cm.Set2(np.linspace(0, 1, len(labels)))
        wedges, texts, autotexts = ax.pie(
            sizes,
            labels=labels,
            autopct="%1.0f%%",
            colors=wedge_colors,
            startangle=90,
            textprops={"fontsize": 10},
        )
        ax.set_title(f"Units Created ({sum(sizes)})")
    else:
        ax.text(0.5, 0.5, "No units\ncreated", ha="center", va="center", fontsize=14, color="#999")
        ax.set_title("Units Created")

    plot_path = CHARTS_DIR / f"replay_episode_{idx}.png"
    fig.savefig(str(plot_path), dpi=150, bbox_inches="tight")
    print(f"Saved: {plot_path}")
    plt.show()


def plot_episode_comparison(episodes):
    """Cross-episode summary: outcomes, action mix, lengths."""
    n = len(episodes)
    if n == 0:
        print("No episodes to compare.")
        return

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle("Cross-Episode Comparison", fontsize=14, fontweight="bold", y=1.02)

    # --- 1. Outcomes & key stats ---
    ax = axes[0]
    labels = [f"Ep {ep.get('episode', i)}" for i, ep in enumerate(episodes)]
    outcomes = [ep["outcome"] for ep in episodes]
    colors = [{"win": "#4CAF50", "loss": "#F44336", "draw": "#FF9800"}[o] for o in outcomes]
    rewards = [ep["total_reward"] for ep in episodes]
    bars = ax.bar(labels, rewards, color=colors, edgecolor="white", linewidth=0.5)
    for bar, outcome in zip(bars, outcomes):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            outcome.upper(),
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold",
            color=bar.get_facecolor(),
        )
    ax.set_ylabel("Total Reward")
    ax.set_title("Outcome & Reward")
    ax.axhline(0, color="grey", linewidth=0.5, linestyle="--")
    ax.grid(True, alpha=0.2, axis="y")

    # --- 2. Action distribution stacked bar ---
    ax = axes[1]
    all_actions = set()
    for ep in episodes:
        all_actions.update(s["action"] for s in ep["steps"])
    actions_ordered = [a for a in ORDERED_ACTIONS if a in all_actions]

    bottoms = np.zeros(n)
    for action in actions_ordered:
        vals = []
        for ep in episodes:
            total = len(ep["steps"])
            cnt = sum(1 for s in ep["steps"] if s["action"] == action)
            vals.append(cnt / total * 100 if total > 0 else 0)
        ax.bar(
            labels,
            vals,
            bottom=bottoms,
            color=ACTION_COLORS.get(action, "#607D8B"),
            label=action,
            edgecolor="white",
            linewidth=0.3,
        )
        bottoms += vals
    ax.set_ylabel("Action %")
    ax.set_title("Action Mix")
    ax.legend(fontsize=7, loc="upper right", ncol=2)

    # --- 3. Episode length & turns ---
    ax = axes[1]  # reuse handled below
    ax2 = axes[2]
    lengths = [ep["length"] for ep in episodes]
    turn_counts = [ep["steps"][-1]["game_state"]["turn"] if ep["steps"] else 0 for ep in episodes]
    x = np.arange(n)
    w = 0.35
    ax2.bar(x - w / 2, lengths, w, color="#2196F3", label="Steps")
    ax2.bar(x + w / 2, turn_counts, w, color="#FF9800", label="Turns")
    ax2.set_xticks(x)
    ax2.set_xticklabels(labels)
    ax2.set_ylabel("Count")
    ax2.set_title("Episode Length")
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.2, axis="y")

    plot_path = CHARTS_DIR / "replay_comparison.png"
    fig.tight_layout()
    fig.savefig(str(plot_path), dpi=150, bbox_inches="tight")
    print(f"Saved: {plot_path}")
    plt.show()


print("Visualization functions defined.")

In [ ]:
# Plot each episode dashboard
for ep in replay_episodes:
    plot_episode(ep)

# Cross-episode comparison
plot_episode_comparison(replay_episodes)

## 9g. Mask coverage and reward decomposition

Two diagnostics that explain *why* training is (or isn't) working:

- **Mask coverage**: distribution of `n_legal_actions` per step. With
  `flat_discrete` action space and `max_flat_actions=512`, only a small
  subset of output neurons is ever legal. If the typical mask has only
  a handful of valid bits, gradient signal is diluted across hundreds
  of dead neurons and `approx_kl` stays near zero.
- **Reward decomposition**: per-episode breakdown of total reward into
  its sources — *action* (per-action shaping like kill/seize), *shaping
  delta* (potential-based ΔΦ), *invalid penalty*, and *terminal*
  (win/loss/draw). If "action" + "shaping" dominate and "terminal" is
  zero, the agent is sitting in a kill-farm local optimum and never
  experiences win/loss signals.

In [ ]:
# Mask coverage + reward decomposition diagnostics across replay episodes.
import matplotlib.pyplot as plt
import numpy as np

REWARD_COMPONENTS = ("action", "shaping_delta", "invalid_penalty", "terminal")
COMPONENT_COLORS = {
    "action": "#4caf50",  # green — per-action shaping
    "shaping_delta": "#2196f3",  # blue  — potential-based ΔΦ
    "invalid_penalty": "#9e9e9e",  # grey  — penalty
    "terminal": "#ff9800",  # orange — win/loss/draw
}


def _episode_mask_counts(ep):
    return [s.get("n_legal_actions", 0) for s in ep["steps"] if s.get("n_legal_actions")]


def _episode_reward_components(ep):
    """Sum each reward component across all steps of an episode."""
    totals = {k: 0.0 for k in REWARD_COMPONENTS}
    for s in ep["steps"]:
        b = s.get("reward_breakdown", {})
        for k in REWARD_COMPONENTS:
            totals[k] += float(b.get(k, 0.0))
    return totals


def plot_mask_coverage_and_reward_decomposition(episodes, max_actions=None):
    """Two-panel diagnostic across all replayed episodes:
    (1) histogram of legal-action counts per step, pooled across episodes,
    (2) stacked bar of reward components per episode (signed).

    Args:
        episodes: list of episode dicts produced by evaluate_with_replay().
        max_actions: optional max_flat_actions for context line on the histogram.
    """
    if not episodes:
        print("No replay episodes — nothing to plot.")
        return

    pooled = []
    for ep in episodes:
        pooled.extend(_episode_mask_counts(ep))

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

    # --- Panel 1: mask coverage histogram ---
    ax = axes[0]
    if pooled:
        max_bin = max(pooled)
        bins = np.arange(0, max_bin + 2) - 0.5
        ax.hist(pooled, bins=bins, color="#1976d2", edgecolor="white")
        median = int(np.median(pooled))
        mean = float(np.mean(pooled))
        ax.axvline(median, color="black", linestyle="--", linewidth=1, label=f"median {median}")
        ax.axvline(mean, color="red", linestyle=":", linewidth=1, label=f"mean {mean:.1f}")
        if max_actions is not None:
            ax.axvline(
                max_actions,
                color="darkred",
                linestyle="-",
                linewidth=1,
                alpha=0.6,
                label=f"max_flat_actions = {max_actions}",
            )
        ax.legend(fontsize=9)
        ax.set_xlabel("Legal actions per step")
        ax.set_ylabel("Step count")
        ax.set_title(f"Mask coverage (pooled across {len(episodes)} episodes)")
    else:
        ax.text(0.5, 0.5, "no n_legal_actions in replay", ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()

    # --- Panel 2: per-episode reward decomposition ---
    ax = axes[1]
    ep_indices = list(range(len(episodes)))
    components = {k: [] for k in REWARD_COMPONENTS}
    for ep in episodes:
        totals = _episode_reward_components(ep)
        for k in REWARD_COMPONENTS:
            components[k].append(totals[k])

    # Plot positive and negative parts separately so signed contributions
    # stack correctly above and below zero.
    pos_bottom = np.zeros(len(ep_indices))
    neg_bottom = np.zeros(len(ep_indices))
    for k in REWARD_COMPONENTS:
        vals = np.array(components[k])
        pos = np.where(vals > 0, vals, 0.0)
        neg = np.where(vals < 0, vals, 0.0)
        ax.bar(ep_indices, pos, bottom=pos_bottom, color=COMPONENT_COLORS[k], label=k, edgecolor="white")
        ax.bar(ep_indices, neg, bottom=neg_bottom, color=COMPONENT_COLORS[k], edgecolor="white")
        pos_bottom += pos
        neg_bottom += neg

    # Overlay actual total reward as a marker for sanity-check
    totals_observed = [ep["total_reward"] for ep in episodes]
    ax.scatter(ep_indices, totals_observed, color="black", marker="D", s=30, zorder=5, label="total (observed)")

    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xticks(ep_indices)
    ax.set_xticklabels([f"Ep {i}\n{ep['outcome']}" for i, ep in enumerate(episodes)])
    ax.set_ylabel("Reward")
    ax.set_title("Reward decomposition by episode")
    ax.legend(fontsize=8, ncol=2, loc="best")

    fig.tight_layout()
    plot_path = CHARTS_DIR / "mask_coverage_and_reward_decomposition.png"
    fig.savefig(str(plot_path), dpi=150, bbox_inches="tight")
    print(f"Saved: {plot_path}")
    plt.show()

    # --- Print numeric summary ---
    print("Mask coverage:")
    if pooled:
        print(f"  steps recorded: {len(pooled)}")
        print(f"  min={min(pooled)}  median={int(np.median(pooled))}  mean={np.mean(pooled):.1f}  max={max(pooled)}")
        if max_actions is not None and pooled:
            usage = np.mean(pooled) / max_actions
            print(f"  mean / max_flat_actions = {usage:.2%}  (low values dilute policy gradient)")
    print("\nReward decomposition (per episode):")
    header = f"  {'ep':>3}  {'outcome':>7}  {'action':>10}  {'shaping':>10}  {'invalid':>10}  {'terminal':>10}  {'total':>10}"
    print(header)
    for i, ep in enumerate(episodes):
        t = _episode_reward_components(ep)
        total_calc = sum(t.values())
        print(
            f"  {i:>3}  {ep['outcome']:>7}  "
            f"{t['action']:>10.1f}  {t['shaping_delta']:>10.1f}  "
            f"{t['invalid_penalty']:>10.1f}  {t['terminal']:>10.1f}  "
            f"{total_calc:>10.1f}"
        )


plot_mask_coverage_and_reward_decomposition(
    replay_episodes,
    max_actions=replay_data["metadata"].get("max_flat_actions"),
)

## 9f. Watch the Agent Play (Video Replay)

Record the trained agent playing a full game and view it as an inline
MP4 video. This uses **headless rendering** via `pygame-ce` — no display
server required (works on Colab).

The video shows the actual game map with terrain, structures, units,
health bars, and player colours exactly as they appear in the game UI.

In [ ]:
# pygame-ce is required for headless rendering. SDL_VIDEODRIVER=dummy was
# already set in the very first cell of the notebook; setting it again here
# would be a no-op once SDL has picked a backend.
try:
    import pygame

    print(f"pygame version: {pygame.ver}")
except ImportError:
    print("Installing pygame-ce...")
    !pip install -q pygame-ce
    import pygame

    print(f"pygame version: {pygame.ver}")

from reinforcetactics.utils.video import display_video_in_notebook, record_evaluation_to_video

print("Video recording utilities loaded.")


In [ ]:
# Sanity check before recording. If sprites still load with
# USE_PIXEL_ART=False below, the kernel is running stale code —
# Runtime > Disconnect and delete runtime, then re-run from the top.
import os

import pygame

from reinforcetactics.ui.renderer import Renderer

print("SDL_VIDEODRIVER:", os.environ.get("SDL_VIDEODRIVER"))
print("pygame display init:", pygame.display.get_init())

_probe_env = make_maskable_env(map_file=MAP_FILE, opponent=OPPONENT)
_probe_env.reset()
_r = Renderer(_probe_env.unwrapped.game_state, replay_mode=True, headless=True, pixel_art=False)
print("screen:", type(_r.screen).__name__, _r.screen.get_size())
print("tile sprites loaded:", sum(v is not None for v in _r.tile_images.values()))
print("unit sprites loaded:", len(_r.unit_images))
print("animator:", _r.animator)
_probe_env.close()

In [ ]:
# Record one full game of the trained agent
# This creates a fresh environment (not the eval_env used above) to avoid
# state leakage, then runs the model through a complete episode while
# capturing every frame with the headless renderer.

video_env = make_maskable_env(
    map_file=MAP_FILE,
    opponent=OPPONENT,
    max_steps=MAX_STEPS,
    max_turns=MAX_TURNS,
    reward_config=REWARD_CONFIG,
    action_space_type=ACTION_SPACE,
)

video_path = str(BENCHMARK_DIR / "agent_replay.mp4")

# Set to True to render with the bundled pixel-art sprites
# (assets/sprites/) instead of the default coloured rects + unit letters.
USE_PIXEL_ART = False

print("Recording agent gameplay...")
result = record_evaluation_to_video(
    env=video_env,
    model=model,
    output_path=video_path,
    fps=4,  # 4 fps — one action per 0.25s
    max_steps=MAX_STEPS,
    deterministic=True,
    use_pixel_art=USE_PIXEL_ART,
)

video_env.close()

winner_str = {1: "Agent wins", 2: "Opponent wins", None: "Draw"}
print(f"\nGame result:   {winner_str.get(result['winner'], 'Unknown')}")
print(f"Total reward:  {result['total_reward']:.1f}")
print(f"Steps taken:   {result['steps']}")
print(f"Video saved:   {result['video_path']}")

In [ ]:
# Display the video inline in the notebook
# On Colab this embeds the MP4 directly; locally it opens in the browser.
display_video_in_notebook(result["video_path"])

## 10. Save results

In [ ]:
# Save benchmark results as JSON
import subprocess


def _git_commit_id() -> str:
    """Return the current HEAD commit (short hash + dirty flag), or 'unknown'."""
    try:
        sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL, text=True).strip()
        dirty = subprocess.call(["git", "diff", "--quiet", "HEAD"], stderr=subprocess.DEVNULL)
        return f"{sha}-dirty" if dirty else sha
    except (subprocess.CalledProcessError, FileNotFoundError):
        return "unknown"


benchmark_data = {
    "metadata": {
        "date": datetime.now().isoformat(),
        "git_commit": _git_commit_id(),
        "map": MAP_FILE,
        "opponent": OPPONENT,
        "max_steps": MAX_STEPS,
        "max_turns": MAX_TURNS,
        "n_envs": N_ENVS,
        "eval_episodes": EVAL_EPISODES,
        "seed": SEED,
        "device": DEVICE,
        "ppo_config": PPO_CONFIG,
        "reward_config": REWARD_CONFIG,
        "storage": "google_drive" if USE_GOOGLE_DRIVE else "local",
    },
    "results": results,
}

results_path = BENCHMARK_DIR / "benchmark_results.json"
with open(results_path, "w") as f:
    json.dump(benchmark_data, f, indent=2)

print(f"Saved results:  {results_path}")
print(f"Git commit:     {benchmark_data['metadata']['git_commit']}")

# Also save as CSV for easy viewing
csv_path = BENCHMARK_DIR / "benchmark_results.csv"
df.to_csv(csv_path, index=False)
print(f"Saved CSV:      {csv_path}")

# List all saved files
print("\nAll benchmark files:")
for p in sorted(BENCHMARK_DIR.rglob("*")):
    if p.is_file():
        size = p.stat().st_size
        if size > 1024 * 1024:
            size_str = f"{size / 1024 / 1024:.1f} MB"
        elif size > 1024:
            size_str = f"{size / 1024:.1f} KB"
        else:
            size_str = f"{size} B"
        rel = p.relative_to(BENCHMARK_DIR)
        print(f"  {str(rel):40s}  {size_str}")

if USE_GOOGLE_DRIVE:
    print("\nFiles are saved to Google Drive at:")
    print(f"  My Drive/{DRIVE_BASE_DIR}/{RUN_SUBDIR}/{RUN_ID}/")
    print("  These files will persist after the Colab runtime disconnects.")
else:
    print("\nFiles are saved to local storage at:")
    print(f"  {BENCHMARK_DIR.resolve()}")
    print("  WARNING: These files will be lost if the Colab runtime disconnects.")
    print("  Set USE_GOOGLE_DRIVE = True in the storage config cell to persist them.")

## 11. TensorBoard (optional)

Launch TensorBoard to inspect detailed training metrics (loss, entropy, etc.).

In [ ]:
# Uncomment to launch TensorBoard inline:
# %load_ext tensorboard
# %tensorboard --logdir benchmarks/ppo_vs_simplebot/tensorboard

print("To view TensorBoard locally, run:")
print(f"  tensorboard --logdir {BENCHMARK_DIR / 'tensorboard'}")

## 12. Interpreting the results

### What to expect (vs `RandomBot`)

| Timesteps | Expected Win Rate | Notes |
|-----------|-------------------|-------|
| 10K | 10–40% | Agent is mostly random itself, learning basic actions |
| 50K | 40–70% | Agent starts capturing structures and trading favourably |
| 200K | 70–95% | Reliably beats a random opponent on the beginner map |
| 1M | 90–100% | Near-saturated — random no longer challenges the policy |
| 2M | 95–100% | Diminishing returns; switch to `OPPONENT="bot"` instead |

`RandomBot` is intentionally weak (uniform sampling from legal actions),
so a well-trained agent should saturate quickly. Once the curve plateaus
above ~90%, switch the `OPPONENT` setting to `"bot"` (`SimpleBot`) for a
stiffer challenge — expect win rates to start lower and climb more slowly.

**Note:** Exact numbers depend on hardware and random seed. The important
thing is that your curve has a similar *shape* — monotonically increasing
win rate with diminishing returns once the agent dominates.

### If your results differ significantly

- **Much worse:** Check that action masking is working (the agent should
  rarely attempt invalid actions). Verify the map file path is correct.
- **Much better:** You may have found better hyperparameters! Consider
  contributing them back.
- **Unstable (oscillating win rate):** Try reducing the learning rate
  or increasing the batch size.

### Next steps

1. **Graduate the opponent:** Set `OPPONENT = "bot"` once you've saturated `random`
2. **Try different maps:** Larger maps (10×10, 14×14) are harder
3. **Tune hyperparameters:** Adjust `ent_coef`, `learning_rate`, etc.
4. **Self-play training:** See `train/train_self_play.py`
5. **AlphaZero:** See `train/train_alphazero.py` for MCTS-based training

In [ ]:
# Clean up environments
vec_env.close()
for _opp_env in eval_envs.values():
    _opp_env.close()
print("Done.")
